In [98]:
pip install --upgrade --quiet google-cloud-aiplatform[agent_engines,adk] googlemaps requests

In [99]:
import os
import json
import requests
from typing import Tuple, Dict, Any, Optional, Sequence, Callable
import logging
import asyncio

# Import Google ADK and Vertex AI Reasoning Engine
import vertexai
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search
from vertexai.agent_engines import AdkApp

# For safety settings with Vertex AI GenerativeModel
from vertexai.preview.generative_models import GenerativeModel, HarmCategory, HarmBlockThreshold, SafetySetting, GenerationResponse

In [100]:
# --- Configuration ---
PROJECT_ID: str = os.getenv("GOOGLE_CLOUD_PROJECT_ID", "qwiklabs-gcp-04-70cfc463a7f7")
LOCATION: str = os.getenv("GOOGLE_CLOUD_REGION", "us-central1") # e.g., "us-central1"
GOOGLE_MAPS_API_KEY: str = os.getenv("GOOGLE_MAPS_API_KEY", "")

# User-Agent for NWS API requests
NWS_USER_AGENT: str = "MyWeatherApp/1.0"

# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Vertex AI Initialized for Project: {PROJECT_ID}, Location: {LOCATION}")

# --- Set up basic logging for callbacks ---
# Using basicConfig for consistent logging, but relying on print() for critical debug info in callbacks
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

Vertex AI Initialized for Project: qwiklabs-gcp-04-70cfc463a7f7, Location: us-central1


In [101]:
# --- Helper functions for callbacks ---

def _extract_location_from_prompt(prompt: str) -> Optional[str]:
    """
    A simple heuristic to extract a potential city name from a prompt.
    This is a very basic implementation and might not catch all cases.
    """
    prompt_lower = prompt.lower()
    keywords = ["weather in", "forecast for", "temperature in", "how's the weather in"]
    for keyword in keywords:
        if keyword in prompt_lower:
            # Try to get the part after the keyword
            parts = prompt_lower.split(keyword, 1)
            if len(parts) > 1:
                # Take the rest of the string and try to clean it
                location_candidate = parts[1].strip()
                # Remove punctuation or trailing questions
                location_candidate = location_candidate.split('?')[0].split('.')[0].strip()
                return location_candidate
    return None

def _is_location_in_united_states(place_name: str) -> bool:
    """
    Checks if a given place name corresponds to a location within the United States
    using Google Maps Geocoding API.
    """
    if not GOOGLE_MAPS_API_KEY or GOOGLE_MAPS_API_KEY == "YOUR_GOOGLE_MAPS_API_KEY":
        logger.warning("Google Maps API Key is not set. Cannot verify location for US check. Defaulting to True.")
        return True # Default to True if API key is missing to avoid blocking everything

    geocode_url: str = "https://maps.googleapis.com/maps/api/geocode/json"
    params: Dict[str, str] = {
        "address": place_name,
        "key": GOOGLE_MAPS_API_KEY
    }
    try:
        response = requests.get(geocode_url, params=params, timeout=5)
        response.raise_for_status()
        data: Dict[str, Any] = response.json()

        if data["status"] == "OK" and data["results"]:
            for component in data["results"][0]["address_components"]:
                if "country" in component["types"] and component["short_name"] == "US":
                    return True
        return False
    except requests.exceptions.RequestException as e:
        logger.error(f"Error during geocoding API request for US check: {e}. Defaulting to True.")
        return True # Default to True if API call fails
    except json.JSONDecodeError as e:
        logger.error(f"Error decoding geocoding API response for US check: {e}. Defaulting to True.")
        return True # Default to True if JSON decode fails

def _check_for_malicious_input(prompt: str) -> bool:
    """
    Checks for malicious input using Vertex AI Safety Settings.
    """

    # Initialize the Vertex AI Generative Model for safety checks
    safety_model = GenerativeModel("gemini-2.5-flash-lite")

    # Define safety settings. Adjust thresholds as needed.
    safety_settings: Sequence[SafetySetting] = [
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=HarmBlockThreshold.BLOCK_ONLY_HIGH
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=HarmBlockThreshold.BLOCK_ONLY_HIGH
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=HarmBlockThreshold.BLOCK_ONLY_HIGH
        ),
    ]

    try:
        response = safety_model.generate_content(
            prompt,
            safety_settings=safety_settings
        )

        if response.prompt_feedback and response.prompt_feedback.block_reason:
            logger.warning(
                f"Vertex AI Safety blocked prompt due to: "
                f"{response.prompt_feedback.block_reason}. "
                f"Safety ratings: {response.prompt_feedback.safety_ratings}"
            )
            return False # Input is malicious based on Vertex AI Safety
        else:
            logger.info("Vertex AI Safety check passed. Input is not malicious.")
            return True # Input passed Vertex AI Safety check
    except Exception as e:
        logger.error(f"Error during Vertex AI Safety check: {e}. Allowing input to proceed.")
        return True # Default to allowing if the safety check itself fails

In [102]:
# --- Callback Functions ---

def before_model_callback(query: str) -> str:
    """
    Callback function executed before the model processes the user's prompt.
    1. Logs the user prompt.
    2. Ensures the user's location (if specified) is in the United States.
    3. Checks for malicious input using Vertex AI Safety Settings.
    """
    # Removed direct print statements, now using logger.info
    logger.info(f"BEFORE_MODEL_CALLBACK - User Query: '{query}'")

    # 2. Ensure the user's location is in the United States
    location_candidate = _extract_location_from_prompt(query)
    if location_candidate:
        logger.info(f"BEFORE_MODEL_CALLBACK - Extracted location candidate: '{location_candidate}'")
        if not _is_location_in_united_states(location_candidate):
            error_message = f"Location '{location_candidate}' is not in the United States. This agent only provides weather for US locations."
            logger.error(error_message)
            raise ValueError(error_message)
        else:
            logger.info(f"BEFORE_MODEL_CALLBACK - Location '{location_candidate}' confirmed to be in the United States.")
    else:
        logger.info("BEFORE_MODEL_CALLBACK - No specific location found in query for US check. Proceeding.")

    # 3. Check for malicious input using Vertex AI Safety
    if not _check_for_malicious_input(query):
        error_message = "Malicious input detected by Vertex AI Safety. Request blocked."
        logger.error(error_message) # Using logger.error for blocked requests
        raise ValueError(error_message)

    return query # Return the (potentially unchanged) query for model processing

def after_model_callback(query: str, model_response_text: str) -> None:
    """
    Callback function executed after the model generates a response.
    1. Logs the model response.
    """
    # Removed direct print statements, now using logger.info
    logger.info(f"AFTER_MODEL_CALLBACK - Original Query: '{query}'")
    logger.info(f"AFTER_MODEL_CALLBACK - Model Response: '{model_response_text.strip()}'")

In [103]:
# --- Tool Definitions (Python functions) ---

def get_lat_long_from_place(place_name: str) -> Optional[Dict[str, float]]:
    """
    Converts a human-readable place name into its latitude and longitude coordinates
    using the Google Maps Geocoding API.
    """
    if not GOOGLE_MAPS_API_KEY or GOOGLE_MAPS_API_KEY == "YOUR_GOOGLE_MAPS_API_KEY":
        logger.error("Google Maps API Key is not set. Cannot perform geocoding.")
        return None

    geocode_url: str = "https://maps.googleapis.com/maps/api/geocode/json"
    params: Dict[str, str] = {
        "address": place_name,
        "key": GOOGLE_MAPS_API_KEY
    }
    try:
        logger.debug(f"Calling Google Geocoding API for '{place_name}'...") # Replaced print with logger.debug
        response = requests.get(geocode_url, params=params, timeout=10)
        response.raise_for_status()
        data: Dict[str, Any] = response.json()
        logger.debug(f"Google Geocoding API response status: {data['status']}") # Replaced print with logger.debug

        if data["status"] == "OK" and data["results"]:
            location = data["results"][0]["geometry"]["location"]
            logger.debug(f"Found coordinates: Lat={location['lat']}, Long={location['lng']}") # Replaced print with logger.debug
            return {"latitude": location["lat"], "longitude": location["lng"]}
        else:
            logger.error(f"Geocoding failed for '{place_name}': {data.get('error_message', data['status'])}") # Replaced print with logger.error
            return None
    except requests.exceptions.RequestException as e:
        logger.error(f"Request error during geocoding API call: {e}") # Replaced print with logger.error
        return None
    except json.JSONDecodeError as e:
        logger.error(f"JSON decode error from geocoding API response: {e}") # Replaced print with logger.error
        return None
    except Exception as e:
        logger.error(f"UNEXPECTED ERROR in get_lat_long_from_place: {e}") # Replaced print with logger.error
        return None

def get_nws_weather_forecast(latitude: float, longitude: float) -> Optional[Dict[str, Any]]:
    """
    Retrieves the current weather forecast for a given latitude and longitude
    using the National Weather Service (NWS) API.
    """
    base_url: str = "https://api.weather.gov"
    points_url: str = f"{base_url}/points/{latitude},{longitude}"

    headers: Dict[str, str] = {"User-Agent": NWS_USER_AGENT}

    try:
        logger.debug(f"Calling NWS /points API for {latitude},{longitude}...") # Replaced print with logger.debug
        response_points = requests.get(points_url, headers=headers, timeout=10)
        response_points.raise_for_status()
        points_data: Dict[str, Any] = response_points.json()
        logger.debug(f"NWS /points API response: {points_data.get('properties', {}).get('forecast')}") # Replaced print with logger.debug

        forecast_url: Optional[str] = points_data["properties"].get("forecast")
        if not forecast_url:
            logger.error(f"Could not find forecast URL for {latitude},{longitude}.") # Replaced print with logger.error
            return None

        logger.debug(f"Calling NWS forecast API: {forecast_url}...") # Replaced print with logger.debug
        response_forecast = requests.get(forecast_url, headers=headers, timeout=10)
        response_forecast.raise_for_status()
        forecast_data: Dict[str, Any] = response_forecast.json()
        logger.debug(f"NWS forecast API response received (truncated): {str(forecast_data)[:200]}...") # Replaced print with logger.debug
        return forecast_data

    except requests.exceptions.RequestException as e:
        logger.error(f"Request error during NWS API call: {e}") # Replaced print with logger.error
        return None
    except json.JSONDecodeError as e:
        logger.error(f"JSON decode error from NWS API response: {e}") # Replaced print with logger.error
        return None
    except KeyError as e:
        logger.error(f"Missing key in NWS API response: {e}. Full response (truncated): {str(points_data)[:200]}...") # Replaced print with logger.error
        return None
    except Exception as e:
        logger.error(f"UNEXPECTED ERROR in get_nws_weather_forecast: {e}") # Replaced print with logger.error
        return None

def get_weather_summary(place_name: str) -> str:
    """
    Provides a weather summary or alert for a given place name.
    This function internally uses get_lat_long_from_place and get_nws_weather_forecast.

    Args:
        place_name: The name of the location for which to get the weather.

    Returns:
        A string containing the weather summary or an error message.
    """
    location_data: Optional[Dict[str, float]] = get_lat_long_from_place(place_name)
    if not location_data:
        return f"Could not find coordinates for {place_name}. Please try again with a different location."

    latitude: float = location_data["latitude"]
    # FIX: Changed 'lng' to 'longitude' to match the key returned by get_lat_long_from_place
    longitude: float = location_data["longitude"]

    weather_data: Optional[Dict[str, Any]] = get_nws_weather_forecast(latitude, longitude)
    if not weather_data:
        return f"Could not retrieve weather data for {place_name}."

    properties: Dict[str, Any] = weather_data.get("properties", {})
    forecast_periods: Optional[list] = properties.get("periods")

    if forecast_periods:
        current_or_next_period: Dict[str, Any] = forecast_periods[0] # Get the current or next forecast period
        name: str = current_or_next_period.get("name", "N/A")
        temperature: str = str(current_or_next_period.get("temperature", "N/A"))
        temperature_unit: str = current_or_next_period.get("temperatureUnit", "")
        forecast_text: str = current_or_next_period.get("detailedForecast", current_or_next_period.get("shortForecast", "No detailed forecast available."))

        summary: str = (
            f"**Weather for {place_name} ({name} Period):**\n"
            f"- Temperature: {temperature}°{temperature_unit}\n"
            f"- Conditions: {forecast_text}\n"
        )
        return summary
    else:
        return f"No forecast periods found for {place_name}."

In [104]:
# --- Create Sub-Agents and their AdkApps ---

print("Creating sub-agents and their applications...")

# Greeter Agent (unchanged)
greeter_agent: Agent = Agent(
    model="gemini-2.0-flash",
    name="GreeterAgent",
    description="A friendly agent specializing in greetings and introductions.",
    instruction=(
        "You are a GreeterAgent. Your only purpose is to respond to greetings and simple conversational openers concisely and warmly. "
        "Do not use any tools or attempt to engage in any other tasks."
    )
)
greeter_sub_app: AdkApp = AdkApp(agent=greeter_agent)
logger.info("Greeter Agent and App created.")


# Search Agent (Stage 1 of Sequential Workflow)
search_sub_agent: Agent = Agent(
    model="gemini-2.0-flash", # Use a capable model for search query generation
    name="SearchAgent",
    description="A specialized agent that performs Google Searches to find raw data for a given query.",
    instruction=(
        "You are a SearchAgent. You receive the original user's query.\n"
        "Your primary task is to use the Google Search tool to find relevant information for this query. "
        "Formulate precise search queries. Present the raw, unedited text content from the top search results that are relevant to the query. "
        "Do not attempt to summarize or refine the information. Just provide the direct findings from the search."
        "\n\n**HANDOFF INFO: You are passing raw search results to the next stage (CritiqueAgent).**" # Explicit handoff info
    ),
    tools=[
        google_search,
    ],
)
# No separate search_sub_app needed now, as SequentialAgent will run it.


# Critique Agent (Stage 2 of Sequential Workflow)
critique_sub_agent: Agent = Agent(
    model="gemini-2.0-flash", # Use a capable model for critique
    name="CritiqueAgent",
    description="An agent that critiques raw information, suggesting improvements for clarity, conciseness, and completeness.",
    instruction=(
        "You are a CritiqueAgent. You receive the original user's question and raw search results from the previous stage (SearchAgent).\n"
        "Your task is to review the raw search results critically in the context of the original question. "
        "Identify any shortcomings: Is the information too technical? Is it incomplete? Is it repetitive? Is it difficult to understand for a general audience? "
        "Suggest specific, actionable improvements for how to refine the answer to the original question. "
        "Do not provide the answer itself, only the critique and suggestions."
        "\n\n**HANDOFF INFO: You are passing your critique and suggestions to the next stage (RefineAgent).**" # Explicit handoff info
    ),
)
# No separate critique_sub_app needed now


# Refine Agent (Stage 3 of Sequential Workflow)
refine_sub_agent: Agent = Agent(
    model="gemini-2.0-flash", # Use a capable model for refinement
    name="RefineAgent",
    description="An agent that refines and rewrites an answer based on raw information and critique suggestions.",
    instruction=(
        "You are a RefineAgent. You receive the original user's question, raw information (from SearchAgent), and a critique with suggestions (from CritiqueAgent).\n"
        "Your task is to combine these to generate a final, polished, and comprehensive answer to the original question. "
        "Incorporate the critique's suggestions to improve the answer's clarity, conciseness, and relevance to the original question. "
        "The final answer should be easy to understand for a general audience and directly address the user's original query. "
        "Provide only the final answer."
        "\n\n**HANDOFF INFO: You are passing the final refined answer to the Root Agent.**" # Explicit handoff info
    ),
)
# No separate refine_sub_app needed now


# --- Create the Sequential Search and Refinement Pipeline Agent ---
print("Creating Sequential Search and Refinement Pipeline Agent...")
search_refine_pipeline_agent: SequentialAgent = SequentialAgent(
    name="SearchRefinePipeline",
    description="A sequential agent that finds information, critiques it, and then refines the answer.",
    sub_agents=[ # Use 'sub_agents' as per documentation
        search_sub_agent,     # Stage 1: Perform search
        critique_sub_agent,   # Stage 2: Critique results
        refine_sub_agent,     # Stage 3: Refine answer
    ],
)
search_refine_pipeline_app: AdkApp = AdkApp(agent=search_refine_pipeline_agent)
logger.info("Sequential Search and Refinement Pipeline Agent and App created.")

# --- Define Delegation Functions (These will be the Root Agent's tools) ---

async def delegate_to_greeter(user_input: str) -> str:
    """Delegates to the GreeterAgent for greetings."""
    # Changed logger.info to print() for immediate visibility of handoff
    print(f"\n--- Root Agent HANDOFF to GreeterAgent with: '{user_input}' ---")
    full_response = ""
    try:
        async for event in greeter_sub_app.async_stream_query(user_id="delegate_user", message=user_input):
            content = event.get('content')
            if content and isinstance(content, dict) and content.get('parts'):
                for part in content['parts']:
                    if 'text' in part:
                        full_response += part['text']
        return full_response if full_response else "GreeterAgent did not respond."
    except Exception as e:
        logger.error(f"ERROR in delegate_to_greeter: Exception when calling greeter_sub_app.async_stream_query(): {e}")
        return f"Error delegating to GreeterAgent: {e}"


# Function to delegate to the Sequential Search and Refinement Pipeline
async def sequential_search_workflow(user_query_for_search: str) -> str:
    """
    Delegates to the Sequential Search and Refinement Pipeline Agent
    to orchestrate Search -> Critique -> Refine.
    """
    print(f"\n--- Root Agent HANDOFF to Sequential Search Workflow for: '{user_query_for_search}' ---")
    final_response_from_pipeline = "An error occurred during the search and refinement process."
    try:
        # We now call the AdkApp of the SequentialAgent directly
        full_pipeline_output = ""
        # The SequentialAgent will manage the flow and state between its sub_agents
        print("  -> Sequential Search Workflow: Initiating pipeline...")
        async for event in search_refine_pipeline_app.async_stream_query(
            user_id="delegate_user",
            message=user_query_for_search, # This is the initial input to the first stage (SearchAgent)
        ):
            content = event.get('content')
            if content and isinstance(content, dict) and content.get('parts'):
                for part in content['parts']:
                    if 'text' in part:
                        # Print intermediate output as it streams from the sequential pipeline
                        # This includes the handoff info embedded in sub-agent instructions
                        print(f"  -> Pipeline Output Stream: {part['text'].strip()}")
                        full_pipeline_output += part['text']

        final_response_from_pipeline = full_pipeline_output if full_pipeline_output else "Sequential Search Workflow failed to produce a response."
        logger.info(f"  -> Sequential Search Workflow Final Output (full): {final_response_from_pipeline}") # Keep full log

    except Exception as e:
        logger.error(f"ERROR in sequential_search_workflow: An exception occurred during the pipeline execution: {e}")
        final_response_from_pipeline = f"An error occurred during the search and refinement process: {e}"

    return final_response_from_pipeline

Creating sub-agents and their applications...
Creating Sequential Search and Refinement Pipeline Agent...


In [105]:
# --- Create the Agent ---

al_agent: Agent = Agent(
    model="gemini-2.0-flash", # Root agent needs good reasoning for delegation
    name="Al",
    description="A helpful AI assistant that understands user requests and delegates tasks to specialized sub-agents or sequential workflows (like the Search-Critique-Refine pipeline).",
    instruction=(
        "You are Al, the main assistant. Your primary goal is to understand the user's intent and delegate to the most appropriate tool (which could be a sub-agent or a multi-step workflow). "
        "Use the available tools to delegate tasks. "
        "1. If the user's request is a greeting (e.g., 'hello', 'hi', 'how are you?'), call the 'delegate_to_greeter' tool with their full message. "
        "2. If the user's request involves finding general information, facts, definitions, or anything that requires looking something up (e.g., 'what is', 'who is', 'tell me about', 'latest news', 'explain'), call the 'sequential_search_workflow' tool with the full user's request."
        "3. For any other types of requests not covered by the above, apologize and state you can only handle greetings or general information searches."
        "Be sure to accurately rephrase or pass the user's original intent to the chosen tool. Remember, you are orchestrating complex processes, including a search-critique-refine pipeline."
    ),
    tools=[
        delegate_to_greeter,
        sequential_search_workflow,
    ],
)
logger.info("Root Agent 'Al' (Delegator) created.")

# --- AdkApp Initialization (for the Root Agent) ---
root_app: AdkApp = AdkApp(
    agent=al_agent,
)
print("Root App created.")

Root App created.


In [106]:
# --- Helper function to process agent events ---
def process_agent_event(event: Dict[str, Any]) -> str:
    """
    Processes a single event from the agent's async_stream_query, assuming
    it is a dictionary and extracting information from its 'content.parts' structure.
    Returns the concatenated text content.
    """
    content = event.get('content')
    if not content or not isinstance(content, dict):
        return ""

    parts = content.get('parts')
    if not parts or not isinstance(parts, list) or not parts:
        return ""

    text_content = []
    for part in parts:
        if not isinstance(part, dict):
            continue

        if 'text' in part:
            print(part['text']) # Original printing to user
            text_content.append(part['text'])

        # Optional: uncomment for debugging tool calls/responses if needed
        if 'function_call' in part:
            function_call_data = part['function_call']
            if isinstance(function_call_data, dict):
                function_name = function_call_data.get('name')
                function_args = function_call_data.get('args')
                print(f"DEBUG: Root Agent called tool: {function_name} with args: {function_args}")

        if 'function_response' in part:
            function_response_data = part['function_response']
            if isinstance(function_response_data, dict) and 'response' in function_response_data:
                response_result = function_response_data['response'].get('result')
                print(f"DEBUG: Tool response: {str(response_result)[:200]}...") # Truncate for printing
    return "".join(text_content)

In [107]:
# --- User Session and Interaction ---

async def run_root_agent_session() -> None:
    """
    Creates a user session and interacts with the root agent.
    Processes input, displays agent responses in Markdown, and handles
    session termination, integrating before/after callbacks.
    """
    print("\n--- Starting Root Agent Session ---")
    print("Type 'bye' or 'exit' to end the session.")

    user_id: str = "test_user_123"

    while True:
        user_input: str = input("\nYou: ")
        if user_input.lower() in ["bye", "exit", "quit", "goodbye"]:
            print("\nAl: Goodbye! Have a great day!")
            break

        print("Al:")
        processed_input = user_input # Initialize with original input

        try:
            # --- Invoke before_model_callback (global pre-processing) ---
            processed_input = before_model_callback(user_input)

            full_model_response_text = "" # To aggregate streaming response
            async for event in root_app.async_stream_query(
                user_id=user_id,
                message=processed_input, # Use the processed input
            ):
                if isinstance(event, dict):
                    full_model_response_text += process_agent_event(event)
                else:
                    print(f"ERROR: Received non-dictionary event: {event}. Cannot process.")

            # --- Invoke after_model_callback (global post-processing) ---
            if full_model_response_text: # Only log if there was an actual response
                after_model_callback(user_input, full_model_response_text)

        except ValueError as e: # Catch ValueErrors raised by callbacks (e.g., safety, location)
            print(f"Al: An issue occurred with your request: {e}")
        except Exception as e:
            logger.error(f"An unexpected error occurred during root agent interaction: {e}")
            print(f"Al: An unexpected error occurred: {e}")

# To run the async function in a Jupyter notebook
import asyncio
# Uncomment the line below to run the interactive session
# asyncio.run(run_root_agent_session())

In [108]:
# --- Test Code for Multi-Agent Delegation ---

async def test_multi_agent_system() -> None:
    """
    Tests the multi-agent system with various queries to demonstrate delegation
    to the Greeter Agent and the Sequential Search Workflow.
    """
    print("\n--- Testing Multi-Agent System with Sequential Search and Greeter ---")

    queries_to_test: list[str] = [
        "Hello Al!", # Should go to GreeterAgent
        "How are you?", # Should go to GreeterAgent
        "What is the capital of France?", # Should go to Sequential Search Workflow
        "Tell me about the latest advancements in renewable energy.", # Should go to Sequential Search Workflow
        "What is the weather like in New York City?", # Root agent should say it can't handle (as weather agent is not a tool now)
        "Tell me something silly.", # Root agent should say it can't handle
        "Tell me something dangerous.", # Safety check in before_model_callback should catch this
    ]

    for user_query in queries_to_test:
        print(f"\n--- Querying with: '{user_query}' ---")
        print(f"You: {user_query}")
        print("Al:")
        processed_input = user_query # Initialize

        try:
            # --- Invoke before_model_callback (global pre-processing) ---
            processed_input = before_model_callback(user_query)

            full_model_response_text = ""
            async for event in root_app.async_stream_query(
                user_id="test_runner",
                message=processed_input,
            ):
                if isinstance(event, dict):
                    full_model_response_text += process_agent_event(event)
                else:
                    print(f"ERROR: Received non-dictionary event: {event}. Cannot process.")

            # --- Invoke after_model_callback (global post-processing) ---
            if full_model_response_text: # Only log if there was an actual response
                after_model_callback(user_query, full_model_response_text)

        except ValueError as e: # Catch ValueErrors raised by callbacks
            print(f"Al: An issue occurred with your request: {e}")
        except Exception as e:
            logger.error(f"An unexpected error occurred during agent interaction for '{user_query}': {e}")
            print(f"Al: An unexpected error occurred: {e}")
        print("-" * 40)

# Run the test cases
await test_multi_agent_system()


--- Testing Multi-Agent System with Sequential Search and Greeter ---

--- Querying with: 'Hello Al!' ---
You: Hello Al!
Al:
DEBUG: Root Agent called tool: delegate_to_greeter with args: {'user_input': 'Hello Al!'}

--- Root Agent HANDOFF to GreeterAgent with: 'Hello Al!' ---
DEBUG: Tool response: Hey there! How's it going?
...
Hey there! How's it going?

----------------------------------------

--- Querying with: 'How are you?' ---
You: How are you?
Al:
DEBUG: Root Agent called tool: delegate_to_greeter with args: {'user_input': 'How are you?'}

--- Root Agent HANDOFF to GreeterAgent with: 'How are you?' ---
DEBUG: Tool response: I'm doing great, thanks for asking! How can I brighten your day?
...
I'm doing great, thanks for asking! How can I brighten your day?

----------------------------------------

--- Querying with: 'What is the capital of France?' ---
You: What is the capital of France?
Al:
DEBUG: Root Agent called tool: sequential_search_workflow with args: {'user_query_for_